In [1]:
#TASK-1

In [2]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(2026)
n = 90
student_ids = np.arange(3001, 3001 + n)

df_hw3 = pd.DataFrame({
    "Student_ID": student_ids,
    "Department": rng.choice(
        ["AIE", "aie ", "Artificial Intelligence Engineering",
         "SE", "software engineering", "CE", "Computer Eng."],
        size=n,
        replace=True,
        p=[0.30, 0.05, 0.05, 0.25, 0.05, 0.25, 0.05]
    ),
    "Study_Hours": np.round(rng.normal(loc=13, scale=4, size=n), 1),
    "Sleep_Hours": np.round(rng.normal(loc=7, scale=1.2, size=n), 1),
    "Attendance_Rate": np.round(rng.uniform(low=0.45, high=1.00, size=n), 2),
    "LMS_Clicks": rng.poisson(lam=120, size=n),
    "Video_Minutes": np.round(rng.normal(loc=260, scale=80, size=n), 0),
    "Forum_Posts": rng.poisson(lam=4, size=n),
    "Teaching_Method": rng.choice(
        ["Traditional", "Flipped", "Project-Based"],
        size=n,
        replace=True,
        p=[0.35, 0.35, 0.30]
    ),
    "Quiz_Score": np.round(rng.normal(loc=72, scale=12, size=n), 0),
    "Project_Score": np.round(rng.normal(loc=78, scale=10, size=n), 0),
    "Advisor_Email": [f"student{i}@bahcesehir.edu.tr" for i in student_ids]
})

# Missing values 
df_hw3.loc[[2, 17, 46], "Study_Hours"] = np.nan
df_hw3.loc[[10, 51], "Sleep_Hours"] = np.nan
df_hw3.loc[[6, 32], "Attendance_Rate"] = np.nan
df_hw3.loc[20, "Quiz_Score"] = np.nan
df_hw3.loc[43, "Project_Score"] = np.nan

# Inconsistent labels ve invalid strings ekle
df_hw3.loc[[4, 24, 43], "Department"] = ["aie ", "SE ", "computer eng."]
df_hw3.loc[[8, 38, 69], "Teaching_Method"] = ["flipped ", "PROJECT-based", "traditional"]
df_hw3.loc[[11, 35], "Advisor_Email"] = [
    "student3012[at]bahcesehir.edu.tr", "student3036@invalid"
]

# Logically impossible / extreme values 
df_hw3.loc[79, "Student_ID"] = df_hw3.loc[9, "Student_ID"]  # Duplicate
df_hw3.loc[15, "Attendance_Rate"] = 1.25  # > 1
df_hw3.loc[27, "Study_Hours"] = -4  # Negative
df_hw3.loc[40, "Sleep_Hours"] = 18  # Extreme
df_hw3.loc[63, "Quiz_Score"] = 132  # > 100
df_hw3.loc[72, "Project_Score"] = -15  # Negative
df_hw3.loc[54, "Video_Minutes"] = 980  # Extreme

# Overall_Score ve Success_Status 
df_hw3["Overall_Score"] = np.round(
    0.40 * df_hw3["Quiz_Score"] +
    0.40 * df_hw3["Project_Score"] +
    0.20 * (100 * df_hw3["Attendance_Rate"]), 0
)
df_hw3["Success_Status"] = np.where(
    df_hw3["Overall_Score"] >= 75, "Successful", "At_Risk"
)

# Duplicated records 
df_hw3 = pd.concat([df_hw3, df_hw3.loc[[5, 13]]], ignore_index=True)

print(df_hw3)

    Student_ID                           Department  Study_Hours  Sleep_Hours  \
0         3001                                  AIE         14.9          8.6   
1         3002                                   SE         14.5          6.0   
2         3003                                   SE          NaN          4.9   
3         3004  Artificial Intelligence Engineering         10.3          8.7   
4         3005                                 aie          12.2          6.5   
..         ...                                  ...          ...          ...   
87        3088                                   SE         14.4          7.1   
88        3089                                   SE         15.3          4.9   
89        3090                                   CE         10.3          4.8   
90        3006                                   CE          9.6          7.1   
91        3014                                   CE         11.5          7.9   

    Attendance_Rate  LMS_Cl

In [3]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.cluster import KMeans
import re
import warnings
warnings.filterwarnings('ignore')
print("done")

#Importing releated libraries.

done


In [4]:
df_hw3.head(6)
#First 6 rows as follows

,Student_ID,Department,Study_Hours,Sleep_Hours,Attendance_Rate,LMS_Clicks,Video_Minutes,Forum_Posts,Teaching_Method,Quiz_Score,Project_Score,Advisor_Email,Overall_Score,Success_Status
0,3001,AIE,14.9,8.6,0.68,110,338.0,2,Traditional,75.0,72.0,student3001@bahcesehir.edu.tr,72.0,At_Risk
1,3002,SE,14.5,6.0,0.87,103,151.0,8,Traditional,75.0,89.0,student3002@bahcesehir.edu.tr,83.0,Successful
2,3003,SE,NaN,4.9,0.54,119,110.0,0,Flipped,67.0,111.0,student3003@bahcesehir.edu.tr,82.0,Successful
3,3004,Artificial Intelligence Engineering,10.3,8.7,0.51,107,256.0,2,Project-Based,76.0,79.0,student3004@bahcesehir.edu.tr,72.0,At_Risk
4,3005,aie,12.2,6.5,0.83,115,109.0,2,Flipped,70.0,83.0,student3005@bahcesehir.edu.tr,78.0,Successful
5,3006,CE,9.6,7.1,0.94,125,238.0,8,Flipped,63.0,69.0,student3006@bahcesehir.edu.tr,72.0,At_Risk


In [5]:
df_hw3.info()
# To understand dtype & Non-Null Count

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 92 entries, 0 to 91
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Student_ID       92 non-null     int64  
 1   Department       92 non-null     object 
 2   Study_Hours      89 non-null     float64
 3   Sleep_Hours      90 non-null     float64
 4   Attendance_Rate  90 non-null     float64
 5   LMS_Clicks       92 non-null     int64  
 6   Video_Minutes    92 non-null     float64
 7   Forum_Posts      92 non-null     int64  
 8   Teaching_Method  92 non-null     object 
 9   Quiz_Score       91 non-null     float64
 10  Project_Score    91 non-null     float64
 11  Advisor_Email    92 non-null     object 
 12  Overall_Score    88 non-null     float64
 13  Success_Status   92 non-null     object 
dtypes: float64(7), int64(3), object(4)
memory usage: 10.2+ KB


In [6]:
df_hw3.describe()
#statistical summary

,Student_ID,Study_Hours,Sleep_Hours,Attendance_Rate,LMS_Clicks,Video_Minutes,Forum_Posts,Quiz_Score,Project_Score,Overall_Score
count,92.000000,89.000000,90.000000,90.000000,92.000000,92.000000,92.000000,91.000000,91.000000,88.000000
mean,3043.967391,13.110112,7.204444,0.747222,119.032609,271.945652,4.206522,73.285714,76.241758,74.761364
std,26.342816,4.855293,1.707940,0.165190,11.191098,111.307155,2.161288,12.664285,13.919084,8.470360
min,3001.000000,-4.000000,2.500000,0.460000,89.000000,97.000000,0.000000,40.000000,-15.000000,34.000000
25%,3020.750000,9.600000,6.325000,0.622500,110.000000,212.250000,2.750000,65.500000,70.000000,71.000000
50%,3043.500000,13.500000,7.100000,0.735000,120.000000,265.500000,4.000000,73.000000,78.000000,74.000000
75%,3066.250000,16.200000,7.900000,0.880000,126.000000,317.250000,6.000000,78.000000,82.000000,79.250000
max,3090.000000,24.700000,18.000000,1.250000,147.000000,980.000000,9.000000,132.000000,111.000000,102.000000


In [7]:
df_hw3.shape

(92, 14)

In [8]:
df_hw3.isnull().sum()
# To see how many NaN var. in each column

'''
Study_Hours        3
Sleep_Hours        2
Attendance_Rate    2
Quiz_Score         1
Project_Score      1
Overall_Score      4
'''

'\nStudy_Hours        3\nSleep_Hours        2\nAttendance_Rate    2\nQuiz_Score         1\nProject_Score      1\nOverall_Score      4\n'

In [9]:
duplicated_id = df_hw3[df_hw3["Student_ID"].duplicated()]
duplicated_id

#These three Student_ID is repeted more than once. I'II deal with it later.

,Student_ID,Department,Study_Hours,Sleep_Hours,Attendance_Rate,LMS_Clicks,Video_Minutes,Forum_Posts,Teaching_Method,Quiz_Score,Project_Score,Advisor_Email,Overall_Score,Success_Status
79,3010,AIE,16.8,6.7,0.61,121,262.0,1,Flipped,65.0,68.0,student3080@bahcesehir.edu.tr,65.0,At_Risk
90,3006,CE,9.6,7.1,0.94,125,238.0,8,Flipped,63.0,69.0,student3006@bahcesehir.edu.tr,72.0,At_Risk
91,3014,CE,11.5,7.9,0.95,126,497.0,8,Flipped,72.0,78.0,student3014@bahcesehir.edu.tr,79.0,Successful


In [10]:
print(df_hw3["Department"].unique())
print()
print()
print(df_hw3["Teaching_Method"].unique())

#I see the dilemma in this cell. For same purpose we have different derivatives. I'II deal with it later.

['AIE' 'SE' 'Artificial Intelligence Engineering' 'aie ' 'CE'
 'software engineering' 'Computer Eng.' 'SE ' 'computer eng.']


['Traditional' 'Flipped' 'Project-Based' 'flipped ' 'PROJECT-based'
 'traditional']


In [11]:
#TASK-2

In [12]:
df_hw3["Department"].unique()

array(['AIE', 'SE', 'Artificial Intelligence Engineering', 'aie ', 'CE',
       'software engineering', 'Computer Eng.', 'SE ', 'computer eng.'],
      dtype=object)

In [13]:
df_hw3["Department"] = df_hw3["Department"].str.strip().str.upper()
#Now every one of them is striped and become capitalized.
df_hw3["Department"].unique()

array(['AIE', 'SE', 'ARTIFICIAL INTELLIGENCE ENGINEERING', 'CE',
       'SOFTWARE ENGINEERING', 'COMPUTER ENG.'], dtype=object)

In [14]:
map_new_name = {
    "AIE" : "AIE",
    "SE" : "SE",
    "ARTIFICIAL INTELLIGENCE ENGINEERING" : "AIE",
    "CE" : "CE",
    "SOFTWARE ENGINEERING" : "SE",
    "COMPUTER ENG." : "CE"
}

df_hw3["Department"] = df_hw3["Department"].map(map_new_name)
df_hw3["Department"].unique()

#I created map and assigned each department to corresponded acronyms.

array(['AIE', 'SE', 'CE'], dtype=object)

In [15]:
target_ids = [3010, 3006, 3014]
duplicated_ids = df_hw3[df_hw3["Student_ID"].isin(target_ids)]
duplicated_ids

#As i said before there are 3 student_id has 2 repetitions. These are listed in below.
#We should not keep duplicates because our model will be wrong cannot create mean from data.
#What is solution? -> .duplicates() has a arguement for repetitions. [First,Last] we choose which to keep and maintain preprocessing.

,Student_ID,Department,Study_Hours,Sleep_Hours,Attendance_Rate,LMS_Clicks,Video_Minutes,Forum_Posts,Teaching_Method,Quiz_Score,Project_Score,Advisor_Email,Overall_Score,Success_Status
5,3006,CE,9.6,7.1,0.94,125,238.0,8,Flipped,63.0,69.0,student3006@bahcesehir.edu.tr,72.0,At_Risk
9,3010,AIE,5.8,8.3,0.74,140,302.0,2,Project-Based,76.0,75.0,student3010@bahcesehir.edu.tr,75.0,Successful
13,3014,CE,11.5,7.9,0.95,126,497.0,8,Flipped,72.0,78.0,student3014@bahcesehir.edu.tr,79.0,Successful
79,3010,AIE,16.8,6.7,0.61,121,262.0,1,Flipped,65.0,68.0,student3080@bahcesehir.edu.tr,65.0,At_Risk
90,3006,CE,9.6,7.1,0.94,125,238.0,8,Flipped,63.0,69.0,student3006@bahcesehir.edu.tr,72.0,At_Risk
91,3014,CE,11.5,7.9,0.95,126,497.0,8,Flipped,72.0,78.0,student3014@bahcesehir.edu.tr,79.0,Successful


In [16]:
df_hw3 = df_hw3.drop_duplicates(["Student_ID"], keep = "first")

#This code line deletes 3 rows from df thats why we assigned to the original df.
#What would happen if we accidently assign to the df_hw3["Student_ID"] -> student_id expecting 100 rows not 95.
#But when we assign to the df_hw3 the new df becomes the one who has 95 rows.

In [17]:
df_hw3.shape
#The row count has decreased while we execute the code it was 92 now its 89. Exactly 3 rows deleted.

(89, 14)

In [18]:
df_hw3["Study_Hours"].unique()

array([14.9, 14.5,  nan, 10.3, 12.2,  9.6, 15.7, 15.4,  5.2,  5.8,  7.9,
       13.5, 21.1, 11.5, 14. ,  8.7,  8.8, 12.9, 16.8, 11.6, 18.6, 13.8,
       15.1, 17.6, -4. ,  3.8, 10.6, 19.7,  8.6, 16.1, 14.8, 17.5, 11.2,
       17.2, 15.9, 19.5, 11.9,  4.9, 16.3,  7.1, 21.4, 16.4, 20.2,  9. ,
        3.2, 19.1,  7.3, 13.4, 13.9,  9.2, 19.2, 15.2, 12. , 12.6,  4. ,
       15. , 16. , 15.5, 24.7, 23.7,  9.3, 16.2,  9.5, 14.4, 15.3])

In [19]:
impossible_Study_H = df_hw3[df_hw3["Study_Hours"] < 0.0]
impossible_Study_H
#Impossible scenario for Study_Hours.

,Student_ID,Department,Study_Hours,Sleep_Hours,Attendance_Rate,LMS_Clicks,Video_Minutes,Forum_Posts,Teaching_Method,Quiz_Score,Project_Score,Advisor_Email,Overall_Score,Success_Status
27,3028,SE,-4.0,7.2,0.76,135,294.0,0,Flipped,79.0,68.0,student3028@bahcesehir.edu.tr,74.0,At_Risk


In [20]:
df_hw3["Sleep_Hours"].unique()

array([ 8.6,  6. ,  4.9,  8.7,  6.5,  7.1,  9.3,  8.4,  7.8,  8.3,  nan,
        7.9,  6.8,  2.5,  7.7,  6.6,  7.3,  7.2,  5.2,  7.4,  7.5,  7. ,
        8.1,  6.4,  8. , 18. ,  6.2,  6.3,  5.1,  5.7,  6.1,  5.6,  6.7,
        8.9,  8.8,  9.5,  5.5,  7.6,  9.6, 10. ,  8.2,  6.9,  5.4,  5.3,
        4.8])

In [21]:
# For sleep hours i want to create Inter Quantile Range then i will detect outliers and finally eliminate them.

Q1 = df_hw3["Sleep_Hours"].quantile(0.25)
Q3 = df_hw3["Sleep_Hours"].quantile(0.75)

IQR = round(Q3 - Q1, 2)
#Calculatin IQR to eliminate outliers
# IQR = np.float64(1.6)

Low_boundary = round(Q1 - (3*IQR)/2, 2) #3.9
Upper_boundary = round(Q3 + (3*IQR)/2, 2) # 10.3

#Outside of [3.9, 10.3] will be considered as a outlier.
Sleep_H_outliers = df_hw3[(df_hw3["Sleep_Hours"] < Low_boundary) | (df_hw3["Sleep_Hours"] > Upper_boundary)]
Sleep_H_outliers

,Student_ID,Department,Study_Hours,Sleep_Hours,Attendance_Rate,LMS_Clicks,Video_Minutes,Forum_Posts,Teaching_Method,Quiz_Score,Project_Score,Advisor_Email,Overall_Score,Success_Status
15,3016,CE,8.7,2.5,1.25,123,104.0,6,Traditional,94.0,70.0,student3016@bahcesehir.edu.tr,91.0,Successful
40,3041,SE,17.2,18.0,0.50,106,97.0,6,Traditional,73.0,71.0,student3041@bahcesehir.edu.tr,68.0,At_Risk


In [22]:
Attendance_R_outliers = df_hw3[(df_hw3["Attendance_Rate"] < 0.00) | (df_hw3["Attendance_Rate"] > 1.00)]
#I used same strategy like the last cell.
Attendance_R_outliers

,Student_ID,Department,Study_Hours,Sleep_Hours,Attendance_Rate,LMS_Clicks,Video_Minutes,Forum_Posts,Teaching_Method,Quiz_Score,Project_Score,Advisor_Email,Overall_Score,Success_Status
15,3016,CE,8.7,2.5,1.25,123,104.0,6,Traditional,94.0,70.0,student3016@bahcesehir.edu.tr,91.0,Successful


In [23]:
Quiz_S_outliers = df_hw3[(df_hw3["Quiz_Score"] < 0.0) | (df_hw3["Quiz_Score"] > 100.0)]
#I used same strategy like the last cell.
Quiz_S_outliers

,Student_ID,Department,Study_Hours,Sleep_Hours,Attendance_Rate,LMS_Clicks,Video_Minutes,Forum_Posts,Teaching_Method,Quiz_Score,Project_Score,Advisor_Email,Overall_Score,Success_Status
25,3026,AIE,14.9,7.1,0.55,131,238.0,5,Project-Based,102.0,80.0,student3026@bahcesehir.edu.tr,84.0,Successful
63,3064,AIE,19.2,7.8,0.53,112,154.0,3,Project-Based,132.0,96.0,student3064@bahcesehir.edu.tr,102.0,Successful


In [24]:
Project_S_outliers = df_hw3[(df_hw3["Project_Score"] < 0.0) | (df_hw3["Project_Score"] > 100.0)]
#I used same strategy like the last cell.
Project_S_outliers

,Student_ID,Department,Study_Hours,Sleep_Hours,Attendance_Rate,LMS_Clicks,Video_Minutes,Forum_Posts,Teaching_Method,Quiz_Score,Project_Score,Advisor_Email,Overall_Score,Success_Status
2,3003,SE,NaN,4.9,0.54,119,110.0,0,Flipped,67.0,111.0,student3003@bahcesehir.edu.tr,82.0,Successful
72,3073,CE,7.3,7.8,0.85,141,311.0,7,Flipped,58.0,-15.0,student3073@bahcesehir.edu.tr,34.0,At_Risk


In [25]:
# For Video_Minutes i want to create Inter Quantile Range and i will detect outliers and finally eliminate them.

Q1 = df_hw3["Video_Minutes"].quantile(0.25)
Q3 = df_hw3["Video_Minutes"].quantile(0.75)

IQR = round(Q3 - Q1, 2)
#Calculatin IQR to eliminate outliers
# IQR = 107.0

Low_boundary = round(Q1 - (3*IQR)/2, 2) #49.5
Upper_boundary = round(Q3 + (3*IQR)/2, 2) # 477.5

#Outside of [49.5, 477.5] will be considered as a outlier.
Video_Minutes_outliers = df_hw3[(df_hw3["Video_Minutes"] < Low_boundary) | (df_hw3["Video_Minutes"] > Upper_boundary)]
Video_Minutes_outliers

,Student_ID,Department,Study_Hours,Sleep_Hours,Attendance_Rate,LMS_Clicks,Video_Minutes,Forum_Posts,Teaching_Method,Quiz_Score,Project_Score,Advisor_Email,Overall_Score,Success_Status
13,3014,CE,11.5,7.9,0.95,126,497.0,8,Flipped,72.0,78.0,student3014@bahcesehir.edu.tr,79.0,Successful
54,3055,SE,20.2,6.1,0.83,119,980.0,4,Traditional,87.0,76.0,student3055@bahcesehir.edu.tr,82.0,Successful


In [52]:
# In here i will handle the outliers.
# First turn into a NaN then impute Median.

outlier_dict = {
    "Study_Hours": impossible_Study_H.index,
    "Sleep_Hours": Sleep_H_outliers.index,
    "Attendance_Rate": Attendance_R_outliers.index,
    "Project_Score": Project_S_outliers.index,
    "Video_Minutes": Video_Minutes_outliers.index,
    "Quiz_Score": Quiz_S_outliers.index
}

for col, idx in outlier_dict.items():
    df_hw3.loc[idx, col] = np.nan

# After assignments all outliers turn into a numpy NaN.

In [53]:
columns_to_fill = ["Video_Minutes", "Study_Hours", "Sleep_Hours", "Attendance_Rate", "Quiz_Score", "Project_Score"]
#creating list for columns make easy to see whats happening.
for col in columns_to_fill:
    df_hw3[col] = df_hw3[col].fillna(np.nanmedian(df_hw3[col]))
    #using for loop enables us to avoid repetitions.
df_hw3.isnull().sum()


#I used median for outliers because in some of the cases outliers might be too high or too low that causes anormal behaviour in mean()
#but median does not affected by outliers.

Student_ID         0
Department         0
Study_Hours        0
Sleep_Hours        0
Attendance_Rate    0
LMS_Clicks         0
Video_Minutes      0
Forum_Posts        0
Teaching_Method    0
Quiz_Score         0
Project_Score      0
Advisor_Email      0
Overall_Score      4
Success_Status     0
dtype: int64

In [55]:
# Verifying that all numeric variables are within valid analytical ranges
columns_to_check = ["Study_Hours", "Sleep_Hours", "Attendance_Rate", "Quiz_Score", "Project_Score", "Video_Minutes"]

for col in columns_to_check:
    print(f"{col}: {df_hw3[col].min()} to {df_hw3[col].max()}")

Study_Hours: 3.2 to 24.7
Sleep_Hours: 4.8 to 10.0
Attendance_Rate: 0.46 to 1.0
Quiz_Score: 40.0 to 98.0
Project_Score: 49.0 to 100.0
Video_Minutes: 97.0 to 466.0


In [58]:
df_hw3["Overall_Score"] = np.round(
    0.40 * df_hw3["Quiz_Score"] +
    0.40 * df_hw3["Project_Score"] +
    0.20 * (100 * df_hw3["Attendance_Rate"]), 0
)
df_hw3["Success_Status"] = np.where(
    df_hw3["Overall_Score"] >= 75, "Successful", "At_Risk"
)
# recomputing overall score 
df_hw3.isnull().sum()

Student_ID         0
Department         0
Study_Hours        0
Sleep_Hours        0
Attendance_Rate    0
LMS_Clicks         0
Video_Minutes      0
Forum_Posts        0
Teaching_Method    0
Quiz_Score         0
Project_Score      0
Advisor_Email      0
Overall_Score      0
Success_Status     0
dtype: int64